In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# Step 2: Load dataset
df = pd.read_csv("SuperMarket Analysis.csv")

# Step 3: Quick EDA
print(df.head())
print(df.info())
print(df.describe())

# Returns a count of empty cells for each column
df.isna().sum()
# Returns rows with one or more empty cells
df[df.isna().any(axis=1)]

# 1. Check if ANY duplicates exist anywhere in the DataFrame
df.duplicated().any()

# 2. Count the total number of duplicate rows
df.duplicated().sum()

# 3. View the actual duplicate rows
df[df.duplicated()]

# 4. View null columns
df.isnull().sum()




In [ ]:
# Step 4: Feature Engineering
# Extract datetime features
df['Date'] = pd.to_datetime(df['Date'])
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Month'] = df['Date'].dt.month

#df['Time'] = pd.to_datetime(df['Time'], format='%H:%M')
df['Time'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p')
df['Hour'] = df['Time'].dt.hour

# Define target and features
X = df.drop(columns=['Sales','Invoice ID', 'Date', 'Time', 'Tax 5%', 'gross income', 'cogs'])
y = df['Sales']


# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=['int64','float64']).columns.tolist()

# Step 5: Preprocessing
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numerical_transformer, numerical_features)
    ])

# Step 6: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 7: Build Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
}

results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}

# Step 8: Compare Results
results_df = pd.DataFrame(results).T
print(results_df)

# Step 9: Visualize Performance
results_df.plot(kind='bar', figsize=(10,6))
plt.title("Model Performance Comparison")
plt.ylabel("Error / Score")
plt.show()
